In [9]:
import pandas as pd
import numpy as np
import os

# ============================================================
# LOAD
# ============================================================
truth = pd.read_parquet(r"raw/truth_match/truth_tract3828.parquet")
obj = pd.read_parquet(r"raw/object_dpdd/object_dpdd_tract3828.parquet")
print(f"truth_match rows: {len(truth):,}")
print(f"object_dpdd rows: {len(obj):,}")


# ============================================================
# CROSS-MATCH
# truth's match_objectId links to object_dpdd's objectId
# ============================================================
merged = truth.merge(obj, left_on="match_objectId", right_on="objectId",
                      how="inner", suffixes=("_true", "_meas"))

n0 = len(merged)
print(f"\n0. After merge: {n0:,} ({100*n0/n0:.1f}%)")


# ============================================================
# CLEANING PIPELINE
# ============================================================

# --- 0.5. Match quality filter (standard DESC DC2 practice) ---
# Removes spurious/ambiguous truth-to-object matches, applied right after
# the merge and before any science cuts.
merged = merged[merged["is_good_match"] == True]
merged = merged[merged["is_nearest_neighbor"] == True]
merged = merged[merged["is_unique_truth_entry"] == True]
print(f"0.5. After match-quality filter: {len(merged):,} ({100*len(merged)/n0:.1f}%)")

# --- 1. Magnitude cut (r < 24.5, matches Duan et al. 2026) ---
merged = merged[merged["mag_r_true"] < 24.5]
print(f"1. After mag cut (r<24.5): {len(merged):,} ({100*len(merged)/n0:.1f}%)")

# --- 2. SNR cut (>= 5, using pipeline's own measured SNR) ---
merged = merged[merged["snr_r_cModel"] >= 5]
print(f"2. After SNR cut (>=5): {len(merged):,} ({100*len(merged)/n0:.1f}%)")

# --- 3. Star/galaxy separation ---
# truth_type codes empirically determined from data:
# 1 = galaxy (redshift spread, real flux)
# 2 = star   (redshift always 0, huge flux)
# 3 = SN     (redshift spread, but static flux_r = 0)
merged = merged[merged["truth_type"] == 1]
print(f"3. After galaxy filter (truth_type==1): {len(merged):,} ({100*len(merged)/n0:.1f}%)")

# --- 4. Quality flags ---
merged = merged[merged["good"] == True]
merged = merged[merged["clean"] == True]
merged = merged[merged["cModelFlux_flag_r"] == False]
print(f"4. After quality flags: {len(merged):,} ({100*len(merged)/n0:.1f}%)")

# --- 5. Drop infinite/NaN positions or errors (garbage fits) ---
merged = merged.replace([np.inf, -np.inf], np.nan)
merged = merged.dropna(subset=["x", "y", "xErr", "yErr", "snr_r_cModel"])
print(f"5. After dropping inf/NaN: {len(merged):,} ({100*len(merged)/n0:.1f}%)")

print(f"\nFinal cleaned dataset: {len(merged):,} sources "
      f"({100*len(merged)/n0:.1f}% of merged, {100*len(merged)/len(truth):.1f}% of original truth catalog)")


# ============================================================
# QUICK CHECK: confirm columns you'll need downstream still exist
# ============================================================
print("\nColumns available for next steps (patch, blendedness, position):")
print([c for c in merged.columns if c in
       ["patch_true", "patch_meas", "ra_true", "ra_meas", "dec_true", "dec_meas",
        "blendedness", "x", "y"]])


# ============================================================
# SAVE
# ============================================================
os.makedirs("processed", exist_ok=True)
merged.to_parquet("processed/cleaned_catalog.parquet")
print(f"\nSaved {len(merged):,} rows to processed/cleaned_catalog.parquet")

truth_match rows: 4,645,754
object_dpdd rows: 1,108,069

0. After merge: 1,108,069 (100.0%)
0.5. After match-quality filter: 970,095 (87.5%)
1. After mag cut (r<24.5): 145,677 (13.1%)
2. After SNR cut (>=5): 145,676 (13.1%)
3. After galaxy filter (truth_type==1): 132,944 (12.0%)
4. After quality flags: 132,837 (12.0%)
5. After dropping inf/NaN: 132,830 (12.0%)

Final cleaned dataset: 132,830 sources (12.0% of merged, 2.9% of original truth catalog)

Columns available for next steps (patch, blendedness, position):
['ra_true', 'dec_true', 'patch_true', 'blendedness', 'dec_meas', 'patch_meas', 'ra_meas', 'x', 'y']

Saved 132,830 rows to processed/cleaned_catalog.parquet
